In [4]:
import numpy as np
import random

# Primary task

In [5]:
def arrivals_in_day(rate, t, Idx_for_process): 
    # Input: rate for arrival time, t is the day in the year, Idx_for_process is the type of patient
    # Output: list of a tuples with patient type in first entry and arrivaltime in the second entry. 


    # Initialize start of day and patients.
    time = 0
    patients = []

    # Let 0 patients arrive if rate is 0
    if rate <=0: 
        return []

    
    while True:
        time += np.random.exponential(1 / rate)

        # Check we are still within one day
        if time > 1:
            break

        # Append patient type and time for arrival
        patients.append((Idx_for_process, t + time))


    return patients

def arrivals_year(lam1, lam2,lam3):
    # Input: lami is the arrival rate function for ward i. 
    # Output: A list of tuples where the first entry in the tuple is the patient type and the last entry is the arrival time.
    
    # Initialize
    t =0
    Patients_1 = []
    Patients_2 = []
    Patients_3 = []

    #Iterate over the days
    while t < 365: 
        # Find rates
        rate1 = lam1(t)
        rate2 = lam2(t)
        rate3 = lam3(t)

        # Simulate arrivals for all three patient types. 
        Patients_1.extend(arrivals_in_day(rate1,t,1))
        Patients_2.extend(arrivals_in_day(rate2,t,2))
        Patients_3.extend(arrivals_in_day(rate3,t,3))

        t+=1

    # Merge list to create one list of all arrivals in a year
    All_patients = sorted(Patients_1 + Patients_2 + Patients_3, key=lambda x: x[1])
    return All_patients


In [6]:
def lam1(t): 
   return -(1/3650)*t**2 + (1/10)*t

def lam2(t): 
   return lam1(t)/5

def lam3(t):
   return 6


X = arrivals_year(lam1,lam2,lam3)

In [7]:
X[-1]

(3, 364.9665285516244)

In [8]:
# System of wards
def system(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and np.mean(bed_frac_i)/bedsi is the mean value of the fraction of beds in use in ward i

    #Initialize
    blocked_A =0
    blocked_B = 0
    blocked_C = 0

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []

    # Iterate through all patients
    for type, t in patientflow_year:
        # Release beds if time has passed of arrivaltime+LOS
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0


        # Find idle beds
        idle_beds_A = np.where(beds_A == 0)[0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        # Append number of beds in use
        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))


        # Patients in ward A
        if type ==1: 
        # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
            # Check for idle beds
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0]
                beds_A[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_A += 1
            
        # Patients in ward B
        elif type ==2: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
            # Check for idle beds
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS

            else:
                # Increase blocked patients in B, and reallocate patient to A.
                blocked_B += 1
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                else: 
                    # If no space in A, randomly choose a patient in A to reallocate
                    blocked_A +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS


        # Patients in ward C
        else: 
            # Find Length-of-Stay
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))
            # Check for idle beds
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC


# Primary Performance Measures 



In [9]:
# Crude monte carlo estimator
def probs(bedsA,bedsB,bedsC,n): #still do not get the input 
    frac_A = []
    frac_B = []
    frac_C = []

    A = []
    B = []
    C = []

    occ_beds_A = []
    occ_beds_B = []
    occ_beds_C = []

    for _ in range(n):
        X = arrivals_year(lam1,lam2,lam3)
        a, b, c, bed_frac_A,bed_frac_B,bed_frac_C = system(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
        
        X = np.array(X)
        type_A = len(np.where(X[:,0] == 1)[0])
        type_B = len(np.where(X[:,0] == 2)[0])
        type_C = len(np.where(X[:,0] == 3)[0])

        frac_A.append(a/type_A)
        frac_B.append(b/type_B)
        frac_C.append(c/type_C)

        occ_beds_A.append(bed_frac_A)
        occ_beds_B.append(bed_frac_B)
        occ_beds_C.append(bed_frac_C)

    A = np.array(A)
    B = np.array(B)
    C = np.array(C)

    
    all = A + B + C
    return np.mean(frac_A),np.mean(frac_B), np.mean(frac_C), np.mean(A),np.mean(B),np.mean(C),np.mean(all),np.mean(occ_beds_A),np.mean(occ_beds_B),np.mean(occ_beds_C)

    

In [10]:
np.random.seed(42)
A = 15
B = 15
C = 45
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

Results for bed distribution A=15, B=15, C=45
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.7544
  Ward B: 0.2120
  Ward C: 0.2132

Average number of relocated patients:
  Ward A: 1675.22
  Ward B: 94.50
  Ward C: 467.54
  Total : 2237.26

Average bed occupancy:
  Ward A: 93.75%
  Ward B: 74.62%
  Ward C: 93.06%


# Sensitivity Analysis

## Control variates - to reduce variance of estimate

In [11]:
# We make a monte carlo estimator just for sum of reallocated patients. 
# This is monte carlo
def sum_relocated(bedsA,bedsB,bedsC,patient_flows):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C.patient_flows is a list of simulated yearly patient flow.
    # Output: mean and variance of the sum of reallocated patients across wards A, B and C
    A = []
    B = []
    C = []

    for X in patient_flows:
        a, b, c, bed_frac_A,bed_frac_B,bed_frac_C = system(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
    all = np.array(A)+np.array(B)+np.array(C)
    return np.mean(all), np.var(all)

In [12]:
# Modify existing system, in order to get output needed for control variate
def system_control(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and LOS_i is the mean value of LOS in use in ward i
    blocked_A =0
    blocked_B = 0
    blocked_C = 0

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    LOS_A = []
    LOS_B = []
    LOS_C = []

    for type, t in patientflow_year:
        # Release beds if time has passed
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0

        idle_beds_A = np.where(beds_A == 0)[0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        if type ==1: 
            LOS = np.random.lognormal(np.log(4*np.sqrt(2)),np.log(2))
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0]
                beds_A[bed_id] = t + LOS
                LOS_A.append(LOS)

            else:
                blocked_A += 1
            

        elif type ==2: 
            LOS = np.random.lognormal(np.log(6*np.sqrt(2)),np.log(2))
    
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS
                LOS_B.append(LOS)

            else:
                blocked_B += 1
                LOS_B.append(LOS)
                # SKal rykke B over i A og hvis A er fuld incremente at nogle er blevet afvist fra A.
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                    
                else: 
                    # Vælger randomly (er det rigtigt???), hvem der skal smides ud af A...
                    blocked_A+=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS
        
        else: 
            LOS = np.random.lognormal(np.log(5*np.sqrt(2)),np.log(2))
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS
                LOS_C.append(LOS)


            else:
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(LOS_A),np.mean(LOS_B), np.mean(LOS_C)

# Control variate function
def control_variate_sum(bedsA, bedsB, bedsC,patient_flows):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. patient_flows is a list of simulated yearly patient flow.
    # Output: mean and variance of the sum of reallocated patients across ward A, B and C. 
    #### Find ci
    # Initialize
    A = []
    B = []
    C = []

    LA = []
    LB = []
    LC = []

    n = int(len(patient_flows)/2)

    # Iterate
    for X in patient_flows[:n]:
        a,b,c,la,lb,lc = system_control(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
        LA.append(la)
        LB.append(lb)
        LC.append(lc)
    
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)
    LA = np.array(LA)
    LB = np.array(LB)
    LC = np.array(LC)

    # Find ci
    ca = -np.cov(A,LA)[0,1]/np.var(LA)
    cB = -np.cov(B,LB)[0,1]/np.var(LB)
    cC = -np.cov(C,LC)[0,1]/np.var(LC)

    # Reinitialize to actually find the control variates
    A = []
    B = []
    C = []

    LA = []
    LB = []
    LC = []

    # Iterate
    for X in patient_flows[n:]:
        a,b,c,la,lb,lc = system_control(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
        LA.append(la)
        LB.append(lb)
        LC.append(lc)
    
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)
    LA = np.array(LA)
    LB = np.array(LB)
    LC = np.array(LC)

    # Find new variable
    Ya = A+ca*(LA-8) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanA = np.mean(Ya)

    YB = B+cB*(LB-12) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanB = np.mean(YB)

    YC = C+cC*(LC-10) #REMEMBER TO CHANGE MEANS IF THEY CHANGE!!!!
    meanC = np.mean(YC)



    return meanA+meanB+meanC, np.var(Ya)+np.var(YB)+np.var(YC)+2*np.cov(Ya,YB)[0,1]+2*np.cov(YB,YC)[0,1]+2*np.cov(Ya,YC)[0,1]

In [19]:
np.random.seed(42)
patient_flow = [arrivals_year(lam1, lam2,lam3) for _ in range(100)]
estimate_MC, var_MC = sum_relocated(15,15,45,patient_flow)
estimate_CV, var_CV = control_variate_sum(15,15,45,patient_flow)

In [20]:
estimate_MC, estimate_CV

(np.float64(2235.71), np.float64(2435.1534343346384))

In [21]:
# Reduction
(var_MC- var_CV)/var_MC*100

np.float64(27.75532590151356)

## Optimization of bed distribution given bed capacity 50, 75 and 100

In [23]:
def generate_grid_starts(bed_cap, patients, step=5, min_beds=1):
    """
    Generate evenly spaced feasible bed distributions.

    Parameters
    ----------
    bed_cap : int
        Total number of beds.
    step : int
        Grid spacing.
    min_beds : int
        Minimum beds allowed in each ward.

    Returns
    -------
    list of tuples
        [(A,B,C), ...]
    """

    starts = []

    for A in range(min_beds, bed_cap - 2*min_beds + 1, step):
        for B in range(min_beds, bed_cap - A - min_beds + 1, step):
            C = bed_cap - A - B

            if C >= min_beds:
                starts.append((A, B, C))
    
    results = []

    i = 0 
    for A, B, C in starts:
        val, var = control_variate_sum(A, B, C, patients)
        results.append(((A, B, C), val))
        print(f"Iteration {i}\{len(starts)}")
        i += 1 
    
    results.sort(key=lambda x: x[1])

    return results[:10]

In [24]:
bed_cap = [50, 75, 100]

patient_flow_100 = [arrivals_year(lam1,lam2,lam3) for _ in range(100)]

best_start_grid = []
for cap in bed_cap:
    best_start_grid.append(generate_grid_starts(cap, patient_flow_100, step=5, min_beds=1))
    print(f"done capacity {cap}")

best_start_grid

Iteration 0\55
Iteration 1\55
Iteration 2\55
Iteration 3\55
Iteration 4\55
Iteration 5\55
Iteration 6\55
Iteration 7\55
Iteration 8\55
Iteration 9\55
Iteration 10\55
Iteration 11\55
Iteration 12\55
Iteration 13\55
Iteration 14\55
Iteration 15\55
Iteration 16\55
Iteration 17\55
Iteration 18\55
Iteration 19\55
Iteration 20\55
Iteration 21\55
Iteration 22\55
Iteration 23\55
Iteration 24\55
Iteration 25\55
Iteration 26\55
Iteration 27\55
Iteration 28\55
Iteration 29\55
Iteration 30\55
Iteration 31\55
Iteration 32\55
Iteration 33\55
Iteration 34\55
Iteration 35\55
Iteration 36\55
Iteration 37\55
Iteration 38\55
Iteration 39\55
Iteration 40\55
Iteration 41\55
Iteration 42\55
Iteration 43\55
Iteration 44\55
Iteration 45\55
Iteration 46\55
Iteration 47\55
Iteration 48\55
Iteration 49\55
Iteration 50\55
Iteration 51\55
Iteration 52\55
Iteration 53\55
Iteration 54\55
done capacity 50
Iteration 0\120
Iteration 1\120
Iteration 2\120
Iteration 3\120
Iteration 4\120
Iteration 5\120
Iteration 6\120
I

[[((31, 16, 3), np.float64(3232.6894978271207)),
  ((16, 16, 18), np.float64(3252.882477737244)),
  ((21, 21, 8), np.float64(3270.789836491611)),
  ((26, 16, 8), np.float64(3284.003035903964)),
  ((11, 16, 23), np.float64(3306.7032619089237)),
  ((21, 11, 18), np.float64(3314.032717980718)),
  ((16, 21, 13), np.float64(3314.5263734197933)),
  ((21, 16, 13), np.float64(3323.574198926076)),
  ((11, 11, 28), np.float64(3327.9747191430415)),
  ((31, 11, 8), np.float64(3346.314435969293))],
 [((36, 21, 18), np.float64(2400.731635292404)),
  ((31, 21, 23), np.float64(2411.1758344433238)),
  ((21, 11, 43), np.float64(2433.7837231145027)),
  ((26, 11, 38), np.float64(2437.7274917282466)),
  ((11, 16, 48), np.float64(2438.1844090356976)),
  ((41, 11, 23), np.float64(2447.873408914852)),
  ((26, 21, 28), np.float64(2449.064920665335)),
  ((51, 11, 13), np.float64(2458.7500216434)),
  ((31, 11, 33), np.float64(2459.3580294281305)),
  ((36, 16, 23), np.float64(2459.7008776539406))],
 [((31, 16, 53

In [25]:
best_start_grid_save = best_start_grid.copy()

In [26]:
# check better in neighboorhood of proposed 
def optimize_beds_stepwise(patient_data, start_dist):
    current_dist = list(start_dist)
    best_dist = list(start_dist)
    
    # Get baseline metrics using your updated function
    best_score, var = control_variate_sum(current_dist[0],current_dist[1], current_dist[2],patient_data)
    print(f"Starting Baseline {current_dist}: {best_score} total issues")

    improved = True
    while improved:
        improved = False
        neighbors = []
        for i in range(3):
            for j in range(3):
                # Ensure we don't drop a ward's bed count below 0
                if i != j and current_dist[i] > 0: 
                    test_dist = list(current_dist)
                    test_dist[i] -= 1
                    test_dist[j] += 1
                    neighbors.append(test_dist)
        
        for neighbor in neighbors:
            print(f"Testing adjustment: {neighbor}...")
            score, var = control_variate_sum(neighbor[0],neighbor[1], neighbor[2],patient_data)
            
            if score < best_score:
                best_score = score
                best_dist = neighbor
                improved = True
        
        if improved:
            current_dist = list(best_dist)
            print(f"Found better distribution: {current_dist} with {best_score} issues")
            
    return best_dist, best_score

In [27]:
# Generate new patients to simulate 

patient_flow_100 = [arrivals_year(lam1,lam2,lam3) for _ in range(100)]

# Search neighboors 
best_overall_grid = []
for dist in best_start_grid: 
    after_search_best_grid = []
    for start_dist_i, _ in dist:
        opt_dist, opt_score = optimize_beds_stepwise(patient_flow_100,  start_dist_i)
        after_search_best_grid.append((opt_dist, opt_score))
        print(f"Start distribution {start_dist_i}")
        print(f"\nFinal Optimal Distribution: {opt_dist} with {opt_score} total problems")
    
    best_overall_grid.append(sorted(after_search_best_grid, key=lambda x: x[1])[0])

Starting Baseline [31, 16, 3]: 3312.697738525939 total issues
Testing adjustment: [30, 17, 3]...
Testing adjustment: [30, 16, 4]...
Testing adjustment: [32, 15, 3]...
Testing adjustment: [31, 15, 4]...
Testing adjustment: [32, 16, 2]...
Testing adjustment: [31, 17, 2]...
Found better distribution: [30, 16, 4] with 3302.5503968295798 issues
Testing adjustment: [29, 17, 4]...
Testing adjustment: [29, 16, 5]...
Testing adjustment: [31, 15, 4]...
Testing adjustment: [30, 15, 5]...
Testing adjustment: [31, 16, 3]...
Testing adjustment: [30, 17, 3]...
Found better distribution: [31, 16, 3] with 3296.0372391623982 issues
Testing adjustment: [30, 17, 3]...
Testing adjustment: [30, 16, 4]...
Testing adjustment: [32, 15, 3]...
Testing adjustment: [31, 15, 4]...
Testing adjustment: [32, 16, 2]...
Testing adjustment: [31, 17, 2]...
Found better distribution: [31, 17, 2] with 3259.5730952663084 issues
Testing adjustment: [30, 18, 2]...
Testing adjustment: [30, 17, 3]...
Testing adjustment: [32, 16,

In [28]:
best_overall_grid_save = best_overall_grid.copy()

In [29]:
# 50 beds performance measures 
A = best_overall_grid[0][0][0]
B = best_overall_grid[0][0][1]
C = best_overall_grid[0][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution capacity 50 beds  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

# 75 beds performance measures 
A = best_overall_grid[1][0][0]
B = best_overall_grid[1][0][1]
C = best_overall_grid[1][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution capacity 75 beds  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

# 100 beds performance measures 
A = best_overall_grid[2][0][0]
B = best_overall_grid[2][0][1]
C = best_overall_grid[2][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs(A,B,C,1000)
print(f"Results for bed distribution capacity 100 beds  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

Results for bed distribution capacity 50 beds  A=27, B=16, C=7
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.5281
  Ward B: 0.1792
  Ward C: 0.8706

Average number of relocated patients:
  Ward A: 1172.71
  Ward B: 79.91
  Ward C: 1908.10
  Total : 3160.72

Average bed occupancy:
  Ward A: 90.10%
  Ward B: 72.91%
  Ward C: 97.89%
Results for bed distribution capacity 75 beds  A=27, B=10, C=38
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.5967
  Ward B: 0.4127
  Ward C: 0.3218

Average number of relocated patients:
  Ward A: 1322.72
  Ward B: 183.91
  Ward C: 705.42
  Total : 2212.05

Average bed occupancy:
  Ward A: 90.58%
  Ward B: 81.77%
  Ward C: 94.80%
Results for bed distribution capacity 100 beds  A=25, B=17, C=58
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.5539
  Ward B: 0.1506
  Ward C: 0.0585

Average number of relocated patients:
  Ward A: 1229.90

## Exponential distribution 

In [30]:
# Exponential length of stay distributial 

def system_exponential(bedsA,bedsB,bedsC, patientflow_year):
    # Input: bedsA is number of beds in ward A,bedsB is number of beds in ward B, bedsC is number of beds in ward C. Patient_flow_year is a list of patients arriving in a year, where each entry in the list is a tuple containing the patient type and their arrival time. 
    # Output: blocked_i is the number of relocated patients for ward i and np.mean(bed_frac_i)/bedsi is the mean value of the fraction of beds in use in ward i

    #Initialize
    blocked_A =0
    blocked_B = 0
    blocked_C = 0
    

    beds_A = np.zeros(bedsA)
    beds_B = np.zeros(bedsB)
    beds_C = np.zeros(bedsC)

    bed_frac_A = []
    bed_frac_B = []
    bed_frac_C = []

    # Iterate through all patients
    for type, t in patientflow_year:
        # Release beds if time has passed of arrivaltime+LOS
        beds_A[beds_A <= t] = 0
        beds_B[beds_B <= t] = 0
        beds_C[beds_C <= t] = 0


        # Find idle beds
        idle_beds_A = np.where(beds_A == 0)[0] #hvad gør det her, hvorfor [0]
        idle_beds_B = np.where(beds_B == 0)[0]
        idle_beds_C = np.where(beds_C == 0)[0]

        # Append number of beds in use
        bed_frac_A.append(bedsA-len(idle_beds_A))
        bed_frac_B.append(bedsB-len(idle_beds_B))
        bed_frac_C.append(bedsC-len(idle_beds_C))


        # Patients in ward A
        if type ==1: 
        # Find Length-of-Stay
            LOS = np.random.exponential(scale=8)
            # Check for idle beds
            if len(idle_beds_A) > 0:
                bed_id = idle_beds_A[0] # samme som anden kommentar
                beds_A[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_A += 1
            
        # Patients in ward B
        elif type ==2: 
            # Find Length-of-Stay
            LOS = np.random.exponential(scale=12)
            # Check for idle beds
            if len(idle_beds_B) > 0:
                bed_id = idle_beds_B[0]
                beds_B[bed_id] = t + LOS

            else:
                # Increase blocked patients in B, and reallocate patient to A.
                blocked_B += 1
                if len(idle_beds_A)>0:
                    bed_id = idle_beds_A[0]
                    beds_A[bed_id] = t + LOS
                else: 
                    # If no space in A, randomly choose a patient in A to reallocate
                    blocked_A +=1
                    bed_id = np.random.choice(len(beds_A))
                    beds_A[bed_id] = t + LOS


        # Patients in ward C
        else: 
            # Find Length-of-Stay
            LOS = np.random.exponential(scale=10)
            # Check for idle beds
            if len(idle_beds_C) > 0:
                bed_id = idle_beds_C[0]
                beds_C[bed_id] = t + LOS

            else:
                # Reallocate patient
                blocked_C += 1
                
    return blocked_A,blocked_B,blocked_C, np.mean(bed_frac_A)/bedsA,np.mean(bed_frac_B)/bedsB,np.mean(bed_frac_C)/bedsC

# Crude monte carlo estimator
def probs_exponential(bedsA,bedsB,bedsC,n): 
    frac_A = []
    frac_B = []
    frac_C = []

    A = []
    B = []
    C = []

    occ_beds_A = []
    occ_beds_B = []
    occ_beds_C = []

    for _ in range(n):
        X = arrivals_year(lam1,lam2,lam3) 
        a, b, c, bed_frac_A,bed_frac_B,bed_frac_C = system_exponential(bedsA,bedsB,bedsC, X)
        A.append(a)
        B.append(b)
        C.append(c)
        X = np.array(X)
        type_A = len(np.where(X[:,0] == 1)[0])
        type_B = len(np.where(X[:,0] == 2)[0])
        type_C = len(np.where(X[:,0] == 3)[0])

        frac_A.append(a/type_A)
        frac_B.append(b/type_B)
        frac_C.append(c/type_C)

        occ_beds_A.append(bed_frac_A)
        occ_beds_B.append(bed_frac_B)
        occ_beds_C.append(bed_frac_C)


    
    all = np.array(A)+np.array(B)+np.array(C)
    return np.mean(frac_A),np.mean(frac_B), np.mean(frac_C), np.mean(A),np.mean(B),np.mean(C),np.mean(all),np.mean(occ_beds_A),np.mean(occ_beds_B),np.mean(occ_beds_C)

In [31]:

# 75 beds exponential distribution performance measure 

A = best_overall_grid[1][0][0]
B = best_overall_grid[1][0][1]
C = best_overall_grid[1][0][2]
fracA,fracB,fracC, meanA, meanB,meanC, meanall,meanbedA,meanbedB,meanbedC = probs_exponential(A,B,C,1000)
print(f"Results for bed distribution capacity 75 beds exponential  A={A}, B={B}, C={C}")
print("-" * 50)

print("Relocation probabilities:")
print(f"  Ward A: {fracA:.4f}")
print(f"  Ward B: {fracB:.4f}")
print(f"  Ward C: {fracC:.4f}")

print("\nAverage number of relocated patients:")
print(f"  Ward A: {meanA:.2f}")
print(f"  Ward B: {meanB:.2f}")
print(f"  Ward C: {meanC:.2f}")
print(f"  Total : {meanall:.2f}")

print("\nAverage bed occupancy:")
print(f"  Ward A: {meanbedA:.2%}")
print(f"  Ward B: {meanbedB:.2%}")
print(f"  Ward C: {meanbedC:.2%}")

Results for bed distribution capacity 75 beds exponential  A=27, B=10, C=38
--------------------------------------------------
Relocation probabilities:
  Ward A: 0.6418
  Ward B: 0.4566
  Ward C: 0.3802

Average number of relocated patients:
  Ward A: 1427.10
  Ward B: 203.25
  Ward C: 832.66
  Total : 2463.01

Average bed occupancy:
  Ward A: 91.60%
  Ward B: 83.56%
  Ward C: 95.70%
